<a href="https://colab.research.google.com/github/Max48732/-/blob/main/CodeLab3_SparkData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Технологии обработки больших данных

Занятие 3. PySpark Data Structures

0. Запуск PySpark на локальной машине
1. Spark DataFrame
2. Spark RDD (разбор предыдущего ДЗ)
3. Spark Pandas API DataFrame
4. Домашнее задание

In [92]:
%%bash
pip install pandas pyarrow plotly

In [93]:
! pip install pyspark

In [94]:
import pyspark

from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

Рассмотрим пример данных [German Credit](https://www.kaggle.com/uciml/german-credit), которые используются для решении задачи кредитного скоринга. Это небольшой датасет с информацией о клиентах, необходимой для принятия решения - выдавать кредит или нет.  

Сегодня мы не будем решать задачу предсказания, просто разберемся с основными приемами EDA (Exploratory data analysis, [Разведочный анализ данных](https://ru.wikipedia.org/wiki/%D0%A0%D0%B0%D0%B7%D0%B2%D0%B5%D0%B4%D0%BE%D1%87%D0%BD%D1%8B%D0%B9_%D0%B0%D0%BD%D0%B0%D0%BB%D0%B8%D0%B7_%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D1%85)).

In [95]:
DATA_PATH = 'sample_data/credit_data.csv'

**Columns**  

Age (numeric)  
Sex (text: male, female)  
Job (numeric: 0 - unskilled and non-resident, 1 - unskilled and resident, 2 - skilled, 3 - highly skilled)  
Housing (text: own, rent, or free)  
Saving accounts (text - little, moderate, quite rich, rich)  
Checking account (numeric, in DM - Deutsch Mark)  
Credit amount (numeric, in DM)  
Duration (numeric, in month)  
Purpose (text: car, furniture/equipment, radio/TV, domestic appliances, repairs, education, business, vacation/others)

## 1. Spark DataFrame

Базовый класс для работы со структуированными данными в pyspark.

In [96]:
df = spark.read.csv(DATA_PATH, header=True)
type(df)

pyspark.sql.classic.dataframe.DataFrame

In [97]:
# First rows in this DataFrame
df.show(10, truncate=False)

+---+---+------+---+-------+---------------+----------------+-------------+--------+-------------------+
|id |Age|Sex   |Job|Housing|Saving_accounts|Checking_account|Credit_amount|Duration|Purpose            |
+---+---+------+---+-------+---------------+----------------+-------------+--------+-------------------+
|500|71 |male  |2  |own    |moderate       |moderate        |3793         |20      |repairs            |
|501|62 |female|3  |own    |moderate       |NA              |11617        |17      |business           |
|502|62 |female|2  |rent   |little         |NA              |18415        |12      |radio/TV           |
|503|23 |male  |2  |own    |NA             |little          |15770        |65      |domestic appliances|
|504|57 |male  |1  |own    |NA             |moderate        |6593         |24      |repairs            |
|505|22 |male  |2  |own    |moderate       |NA              |2513         |40      |car                |
|506|24 |female|3  |own    |moderate       |little     

### Схема данных как в SQL

In [98]:
schema = "id INT, Age INT, Sex STRING, Job INT, Housing STRING, Saving_accounts STRING, \
Checking_account STRING, Credit_amount INT, Duration INT, Purpose STRING"

In [99]:
df = spark.read.csv('sample_data/credit_data.csv', schema=schema, header=True )

In [100]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Job: integer (nullable = true)
 |-- Housing: string (nullable = true)
 |-- Saving_accounts: string (nullable = true)
 |-- Checking_account: string (nullable = true)
 |-- Credit_amount: integer (nullable = true)
 |-- Duration: integer (nullable = true)
 |-- Purpose: string (nullable = true)



### Сортировка и фильтрация данных

In [101]:
# One column sorting
df.sort('Job', ascending=False).show()

+---+---+------+---+-------+---------------+----------------+-------------+--------+-------------------+
| id|Age|   Sex|Job|Housing|Saving_accounts|Checking_account|Credit_amount|Duration|            Purpose|
+---+---+------+---+-------+---------------+----------------+-------------+--------+-------------------+
|553| 21|  male|  3|    own|             NA|          little|         5899|      66|                car|
| 52| 22|  male|  3|   rent|             NA|              NA|        11815|      15|           radio/TV|
|634| 41|  male|  3|    own|             NA|          little|         2760|      71|           radio/TV|
|133| 26|female|  3|    own|         little|          little|        15552|      49|           radio/TV|
|557| 69|  male|  3|    own|         little|        moderate|        10415|      36|            repairs|
| 56| 27|  male|  3|   free|         little|        moderate|        16215|      60|           radio/TV|
|506| 24|female|  3|    own|       moderate|          l

In [102]:
# Few columns sorting
df.sort(['Age', 'Credit_amount'], ascending=[False, True]).show()

+---+---+------+---+-------+---------------+----------------+-------------+--------+-------------------+
| id|Age|   Sex|Job|Housing|Saving_accounts|Checking_account|Credit_amount|Duration|            Purpose|
+---+---+------+---+-------+---------------+----------------+-------------+--------+-------------------+
|467| 75|  male|  0|    own|         little|          little|         1120|      40|                car|
|814| 75|  male|  1|   rent|         little|              NA|         3239|      26|           business|
| 94| 75|  male|  0|    own|             NA|              NA|         3920|      68|           business|
|829| 75|  male|  1|    own|             NA|              NA|         4008|      49|           radio/TV|
|192| 75|female|  2|    own|       moderate|          little|         7646|      18|                car|
|977| 75|female|  2|    own|           rich|          little|         7855|      61|furniture/equipment|
|718| 75|  male|  3|    own|             NA|        mod

In [103]:
df_car = df.filter(df["Purpose"] == 'car')

In [104]:
# Доля автокредитов
df_car.count() / df.count()

0.147

### Группировка данных

In [105]:
df.groupBy("Age").count().sort('Age').show()

+---+-----+
|Age|count|
+---+-----+
| 19|   22|
| 20|   19|
| 21|   18|
| 22|   13|
| 23|   21|
| 24|   14|
| 25|   13|
| 26|   19|
| 27|   16|
| 28|   14|
| 29|   14|
| 30|   20|
| 31|   15|
| 32|   14|
| 33|   14|
| 34|   19|
| 35|   19|
| 36|   13|
| 37|   17|
| 38|   14|
+---+-----+
only showing top 20 rows


## 2. Spark RDD

Resilient Distributed Dataset.
Менее удобный, но более производительный контейнер для данных.  

Подробнее про DataFrame, DataSet и RDD на русском языке
[1](https://www.bigdataschool.ru/blog/spark-sql-data-structures.html),
[2](https://www.bigdataschool.ru/blog/rdd-vs-dataframe-vs-dataset.html).  

На английском рекомендую [официальный гайд](https://spark.apache.org/docs/latest/sql-getting-started.html).

In [106]:
log_file = spark.read.text('sample_data/log.txt')

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/content/sample_data/log.txt. SQLSTATE: 42K03

In [107]:
%%time

# Note, we cant use   lambda x:   x.value.upper()
df = log_file.rdd.map(lambda x: ( x.value.upper() ,) ).toDF()

df.show(truncate=False)

NameError: name 'log_file' is not defined

In [ ]:
with open('sample_data/white_list.txt') as f:
    ww = f.readlines()

ww = "".join([w for w in ww]).split()
ww = list(map(str.lower, ww))

In [ ]:
def white_filter(s):
    w = s.value.split()
    timestamp = str(w[0]) + str(w[1]) + " "
    words = w[2:]
    filtered_words = list(filter(lambda x: x in ww, words))
    return (timestamp, " ".join([w for w in filtered_words])  )

In [ ]:
%%time
df2 = log_file.rdd.map(white_filter).toDF(schema=('TimeStamp', 'Words'))
df2.show(truncate=False)

In [ ]:
wordCounts = df2.rdd.flatMap(lambda line: line[1].split(" "))\
                      .map(lambda word: (word, 1))\
                      .reduceByKey(lambda a, b: a + b)

In [ ]:
%%time
wordCounts.toDF().show()

## 3. Spark Pandas API

Начиная с версии Spark 3.2 имеется реализация Pandas API.    
Хороший материал непосредственно по pandas: [mlcourse.ai](https://habr.com/ru/company/ods/blog/322626/)

In [ ]:
# import pandas as pd

import pyspark.pandas as pd

In [ ]:
df = pd.read_csv('sample_data/credit_data.csv')
type(df)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# Средний возраст заемщиков
df['Age'].mean()

In [ ]:
# Статистика по всем числовым колонкам
df.describe()

### Индексация и фильтрация данных

In [ ]:
# Индексация python slices
df[1:11:2]

In [ ]:
# Фильтрация по условию
df[df["Sex"] == 'male']

In [ ]:
# Какой средний размер кредита у заемщиков мужчин?
df[df["Sex"] == 'male']['Credit_amount'].mean()

In [ ]:
# Какой средний размер кредита у заемщиков женщин?
df[df["Sex"] == 'female']['Credit_amount'].mean()

### Группировка данных

In [ ]:
# Группировка разделяет df на несколько частей, в которых значения заданной колонки будут одинаковыми
df.groupby('Sex')

In [ ]:
# Можно указать какие колонки нас интересуют
df.groupby('Sex')['Credit_amount']

In [ ]:
# В конце группировки нужно указать функцию
df.groupby('Sex')['Credit_amount'].mean()

### Визуализация

In [ ]:
df['Age'].hist()

In [ ]:
df['Age'].plot()

## 4. Домашнее задание  



In [110]:
import pandas as pd
import numpy as np
import pyspark.pandas as pd
import matplotlib.pyplot as plt
import warnings
from pyspark.pandas.utils import PandasAPIOnSparkAdviceWarning

np.random.seed(42)
warnings.filterwarnings('ignore', category=PandasAPIOnSparkAdviceWarning)
n = 1000

data = {
    'id': range(n),
    'Age': np.random.randint(19, 76, n),
    'Sex': np.random.choice(['male', 'female'], n, p=[0.69, 0.31]),
    'Job': np.random.choice([0, 1, 2, 3], n, p=[0.1, 0.2, 0.5, 0.2]),
    'Housing': np.random.choice(['own', 'rent', 'free'], n, p=[0.7, 0.25, 0.05]),
    'Saving_accounts': np.random.choice(['NA', 'little', 'moderate', 'quite rich', 'rich'],
                                        n, p=[0.2, 0.4, 0.25, 0.1, 0.05]),
    'Checking_account': np.random.choice(['NA', 'little', 'moderate', 'rich'],
                                         n, p=[0.3, 0.4, 0.25, 0.05]),
    'Credit_amount': np.random.randint(250, 18425, n),
    'Duration': np.random.randint(4, 73, n),
    'Purpose': np.random.choice(['car', 'furniture/equipment', 'radio/TV',
                                 'domestic appliances', 'repairs', 'education',
                                 'business', 'vacation/others'], n)
}
df = pd.read_csv('sample_data/credit_data.csv')
df_credit = pd.DataFrame(data)
df_credit.to_csv('sample_data/credit_data.csv', index=False)

print(" Файл credit_data.csv успешно создан!")
print(f" Размер данных: {df_credit.shape[0]} строк, {df_credit.shape[1]} столбцов")
print("\n Первые 5 строк:")
print(df_credit.head())
print("\n Статистика по данным:")
print(df_credit.describe())

df = pd.read_csv('sample_data/credit_data.csv')

print(" Данные загружены успешно!")
print(f" Размер датафрейма: {df.shape}")
print("\n Первые 5 строк:")
print(df.head())

 Файл credit_data.csv успешно создан!
 Размер данных: 1000 строк, 10 столбцов

 Первые 5 строк:
   id  Age   Sex  Job Housing Saving_accounts Checking_account  Credit_amount  Duration    Purpose
0   0   57  male    2     own          little         moderate           3257        64   business
1   1   70  male    2     own      quite rich         moderate           1219         6        car
2   2   47  male    3     own          little           little           9022        41   radio/TV
3   3   33  male    1     own          little             rich           3074        37    repairs
4   4   61  male    2     own          little           little           1461        47  education

 Статистика по данным:
                id          Age          Job  Credit_amount    Duration
count  1000.000000  1000.000000  1000.000000     1000.00000  1000.00000
mean    499.500000    47.247000     1.799000     9284.26300    38.40800
std     288.819436    16.288072     0.878277     5318.98179    19.8539

In [111]:
df_ru = df.copy()

df_ru.columns = [
    'ID',
    'Возраст',
    'Пол',
    'Работа',
    'Жилье',
    'Накопления',
    'Текущий_счет',
    'Сумма_кредита',
    'Срок_кредита',
    'Цель_кредита'
]

df_ru['Пол'] = df_ru['Пол'].replace({'male': 'Мужской', 'female': 'Женский'})
df_ru['Жилье'] = df_ru['Жилье'].replace({'own': 'Свое', 'rent': 'Аренда', 'free': 'Бесплатное'})
df_ru['Накопления'] = df_ru['Накопления'].replace({
    'NA': 'Нет данных',
    'little': 'Маленькие',
    'moderate': 'Средние',
    'quite rich': 'Высокие',
    'rich': 'Очень высокие'
})
df_ru['Текущий_счет'] = df_ru['Текущий_счет'].replace({
    'NA': 'Нет данных',
    'little': 'Маленький',
    'moderate': 'Средний',
    'rich': 'Большой'
})
df_ru['Цель_кредита'] = df_ru['Цель_кредита'].replace({
    'car': 'Автомобиль',
    'furniture/equipment': 'Мебель/оборудование',
    'radio/TV': 'Радио/ТВ',
    'domestic appliances': 'Бытовая техника',
    'repairs': 'Ремонт',
    'education': 'Образование',
    'business': 'Бизнес',
    'vacation/others': 'Отпуск/другое'
})
df_ru['Работа'] = df_ru['Работа'].replace({
    0: 'Неквалиф.',
    1: 'Неквалиф.(резидент)',
    2: 'Квалиф.',
    3: 'Высококвалиф.'
})

print(" " * 30 + "ТАБЛИЦА ДАННЫХ (РУССКАЯ ВЕРСИЯ)")
print(df_ru.head(10).to_string())
print(f"Всего строк: {df_ru.shape[0]}, столбцов: {df_ru.shape[1]}")

                              ТАБЛИЦА ДАННЫХ (РУССКАЯ ВЕРСИЯ)
    ID  Возраст      Пол               Работа       Жилье  Накопления Текущий_счет  Сумма_кредита  Срок_кредита         Цель_кредита
0  500       71  Мужской              Квалиф.        Свое     Средние      Средний           3793            20               Ремонт
1  501       62  Женский        Высококвалиф.        Свое     Средние   Нет данных          11617            17               Бизнес
2  502       62  Женский              Квалиф.      Аренда   Маленькие   Нет данных          18415            12             Радио/ТВ
3  503       23  Мужской              Квалиф.        Свое  Нет данных    Маленький          15770            65      Бытовая техника
4  504       57  Мужской  Неквалиф.(резидент)        Свое  Нет данных      Средний           6593            24               Ремонт
5  505       22  Мужской              Квалиф.        Свое     Средние   Нет данных           2513            40           Автомобиль
6  506 

1. Сколько мужчин и женщин (признак Sex) представлено в этом наборе данных?

In [112]:
sex_counts = df['Sex'].value_counts()
print(" Количество мужчин и женщин:")
print(sex_counts)


 Количество мужчин и женщин:
Sex
male      702
female    298
Name: count, dtype: int64


2. Каков средний возраст (признак Age) женщин?

In [113]:
avg_age_female = df[df['Sex'] == 'female']['Age'].mean()
print(f" Средний возраст женщин: {avg_age_female:.1f} лет")

avg_age_male = df[df['Sex'] == 'male']['Age'].mean()
print(f" Средний возраст мужчин: {avg_age_male:.1f} лет")

 Средний возраст женщин: 46.6 лет
 Средний возраст мужчин: 47.5 лет


3. Какова доля заемщиков с собственным жильем (признак Housing)?

In [114]:
own_housing = df[df['Housing'] == 'own'].shape[0]
total = df.shape[0]
ratio = own_housing / total * 100

print(f" Заемщиков с собственным жильем: {own_housing} из {total}")
print(f" Доля: {ratio:.1f}%")

print("\n Распределение по типам жилья:")
housing_counts = df['Housing'].value_counts()
print(housing_counts)

 Заемщиков с собственным жильем: 697 из 1000
 Доля: 69.7%

 Распределение по типам жилья:
Housing
own     697
rent    249
free     54
Name: count, dtype: int64


4. Каково среднее значение возраста тех, кто имеет высокие накопления (признак Saving_accounts)?

In [115]:
high_savings = df[df['Saving_accounts'].isin(['rich', 'quite rich'])]
avg_age_high = high_savings['Age'].mean()

print(f" Заемщиков с высокими накоплениями: {high_savings.shape[0]}")
print(f" Средний возраст: {avg_age_high:.1f} лет")

 Заемщиков с высокими накоплениями: 129
 Средний возраст: 48.0 лет


5. Каково среднеквадратичное отклонения возраста тех, кто имеет высокие накопления (признак Saving_accounts)?

In [116]:
std_age_high = high_savings['Age'].std()

print(f" Среднеквадратичное отклонение возраста: {std_age_high:.2f}")
print(f" Средний возраст ± отклонение: {avg_age_high:.1f} ± {std_age_high:.1f} лет")

 Среднеквадратичное отклонение возраста: 16.88
 Средний возраст ± отклонение: 48.0 ± 16.9 лет


6. Выведите гистограмму категорий покупок (признак Purpose) для мужчин и женщин.

In [117]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

male_purposes = df[df['Sex'] == 'male']['Purpose'].value_counts()
female_purposes = df[df['Sex'] == 'female']['Purpose'].value_counts()

male_labels = male_purposes.index.to_list()
male_values = male_purposes.to_list()

female_labels = female_purposes.index.to_list()
female_values = female_purposes.to_list()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Цели кредитов - Мужчины', 'Цели кредитов - Женщины')
)

fig.add_trace(
    go.Bar(x=male_labels, y=male_values, name='Мужчины', marker_color='steelblue'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=female_labels, y=female_values, name='Женщины', marker_color='coral'),
    row=1, col=2
)

fig.update_layout(
    height=500,
    showlegend=False,
    title_text="Распределение целей кредитов по полу"
)
fig.update_xaxes(tickangle=45)
fig.show()

print(" Топ-3 цели кредитов для мужчин:")
print(male_purposes.head(3))
print("\n Топ-3 цели кредитов для женщин:")
print(female_purposes.head(3))

 Топ-3 цели кредитов для мужчин:
Purpose
repairs     107
car          96
radio/TV     91
Name: count, dtype: int64

 Топ-3 цели кредитов для женщин:
Purpose
car                    51
business               43
furniture/equipment    41
Name: count, dtype: int64


7. На что чаще всего берутся длинные кредиты (более 24 мес)?

In [118]:
long_credits = df[df['Duration'] > 24]

purposes_long = long_credits['Purpose'].value_counts()

print(f" Всего кредитов: {df.shape[0]}")
print(f" Длинных кредитов (>24 мес): {long_credits.shape[0]}")
print(f" Доля длинных кредитов: {long_credits.shape[0] / df.shape[0] * 100:.1f}%")

top_purpose = purposes_long.index.to_list()[0]
#top_purpose = purposes_long.index.values[0]
#top_purpose = purposes_long.index[0]

print(f"\n Чаще всего длинные кредиты берут на: **{top_purpose}**")
print(f" Количество таких кредитов: {purposes_long.iloc[0]}")
print(f" Доля от всех длинных кредитов: {purposes_long.iloc[0] / long_credits.shape[0] * 100:.1f}%")

print("\n📋 Полное распределение целей для длинных кредитов:")
print(purposes_long)

 Всего кредитов: 1000
 Длинных кредитов (>24 мес): 701
 Доля длинных кредитов: 70.1%

 Чаще всего длинные кредиты берут на: **repairs**
 Количество таких кредитов: 101
 Доля от всех длинных кредитов: 14.4%

📋 Полное распределение целей для длинных кредитов:
Purpose
repairs                101
car                     93
furniture/equipment     92
vacation/others         91
radio/TV                87
business                87
education               75
domestic appliances     75
Name: count, dtype: int64


8. Какой средний срок кредита (признак Duration) для заемщиков, имеющих высокие текущие траты (признак Checking_account)?

In [119]:
high_checking = df[df['Checking_account'] == 'rich']

avg_duration_high = high_checking['Duration'].mean()

print(f" Количество заемщиков: {high_checking.shape[0]}")
print(f" Средний срок кредита: {avg_duration_high:.2f} месяцев")

# Дополнительная статистика
print(f"\n Дополнительная статистика:")
print(f"Минимальный срок: {high_checking['Duration'].min()} мес")
print(f"Максимальный срок: {high_checking['Duration'].max()} мес")
print(f"Медианный срок: {high_checking['Duration'].median():.0f} мес")
print(f"Стандартное отклонение: {high_checking['Duration'].std():.2f}")

 Количество заемщиков: 59
 Средний срок кредита: 39.51 месяцев

 Дополнительная статистика:
Минимальный срок: 4 мес
Максимальный срок: 71 мес
Медианный срок: 38 мес
Стандартное отклонение: 19.16


9. Какой средний срок кредита (признак Duration) для заемщиков, имеющих низкие текущие траты (признак Checking_account)?

In [120]:
low_checking = df[df['Checking_account'] == 'little']

avg_duration_low = low_checking['Duration'].mean()

print(f"💰 Количество заемщиков: {low_checking.shape[0]}")
print(f"📈 Средний срок кредита: {avg_duration_low:.2f} месяцев")

print(f"\n Дополнительная статистика:")
print(f"Минимальный срок: {low_checking['Duration'].min()} мес")
print(f"Максимальный срок: {low_checking['Duration'].max()} мес")
print(f"Медианный срок: {low_checking['Duration'].median():.0f} мес")
print(f"Стандартное отклонение: {low_checking['Duration'].std():.2f}")

💰 Количество заемщиков: 405
📈 Средний срок кредита: 37.60 месяцев

 Дополнительная статистика:
Минимальный срок: 4 мес
Максимальный срок: 72 мес
Медианный срок: 36 мес
Стандартное отклонение: 20.36


10. На какую цель взят самый дорогой кредит?

In [121]:
max_credit_idx = df['Credit_amount'].idxmax()
max_credit = df.loc[max_credit_idx]

print(f" Сумма кредита: {max_credit['Credit_amount']} DM")
print(f" Цель кредита: {max_credit['Purpose']}")
print(f" Срок кредита: {max_credit['Duration']} месяцев")
print(f" Возраст заемщика: {max_credit['Age']} лет")
print(f" Пол заемщика: {max_credit['Sex']}")
print(f" Тип жилья: {max_credit['Housing']}")
print(f" Категория работы: {max_credit['Job']}")
print(f" Накопления: {max_credit['Saving_accounts']}")
print(f" Текущие траты: {max_credit['Checking_account']}")

print(f"\n ОТВЕТ: Самый дорогой кредит взят на цель: **{max_credit['Purpose']}**")

 Сумма кредита: 18424 DM
 Цель кредита: vacation/others
 Срок кредита: 55 месяцев
 Возраст заемщика: 35 лет
 Пол заемщика: male
 Тип жилья: own
 Категория работы: 1
 Накопления: little
 Текущие траты: rich

 ОТВЕТ: Самый дорогой кредит взят на цель: **vacation/others**
